# Gene Expression, Gene Symbol Lists, and Expression Statistics Related to TCGA Cancer and Irisin-Associated Pathways Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.r44v-era2/fair2.json) dataset using the `mlcroissant` library. You will:

- Load the dataset via its Croissant schema
- Examine record sets, fields, and IDs
- Extract, process, and visualize the data

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.r44v-era2/fair2.json"

# Load the dataset and its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Let's review all available record sets, their fields, columns, and the corresponding `@id` values for precise referencing during data extraction and analysis.

In [ ]:
# List all record sets with @id, name, and description
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:\n")

for rs in record_sets:
    print(f"RecordSet: @id={rs['@id']} name={rs.get('name', '')}\n  description={rs.get('description', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        print(f"    Field: @id={fld['@id']} name={fld.get('name', '')} (dataType: {fld.get('dataType', 'unknown')})")
        columns = fld.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        for col in columns:
            print(f"       Column: @id={col.get('@id', 'n/a')} name={col.get('name', '')}")
    print()

## 3. Data Extraction
We'll extract data from one or more record sets. Use the discovered record set `@id` values above.

In [ ]:
# List the record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
print("Available record set @id's:")
for rsid in record_set_ids:
    print(f"- {rsid}")# For illustration, let's try to load the first record set (edit as appropriate for analysis)
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nExtracting records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error: {e}")

# Choose a populated record set for further analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nProceeding with main record set: {main_record_set_id}")
    print(f"Available columns: {dataframes[main_record_set_id].columns.tolist()}")
else:
    main_record_set_id = None
    print("No tabular record sets available for EDA.")

## 4. Exploratory Data Analysis (EDA)
Let's perform basic processing on the chosen record set. This will illustrate filtering, normalization, and grouping on a numeric field for demonstration.

_Note: Please update `numeric_field` and `group_field` variables below with the IDs or column names appropriate for your use-case and record set._

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Analyzing record set: {main_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Suggest/guess a numeric field
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int', 'float'] or pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric candidate fields: {numeric_field_candidates}")
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
    else:
        # Manually set if no numeric inferred
        numeric_field = df.columns[1] if len(df.columns) > 1 else df.columns[0]

    print(f"Using numeric field: {numeric_field}")

    # Thresholding/filter
    try:
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nFirst few normalized values:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    except Exception as e:
        print(f"Could not filter/normalize numeric field: {e}")

    # Group by a categorical field if available
    group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f'\nGrouping by field: {group_field}')
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(grouped_df.head())
    else:
        print('No candidate fields for grouping found.')
else:
    print('No suitable record set loaded for EDA.')

## 5. Visualization
Visualizing the data using matplotlib for distribution and relationship exploration.

_If the analysis record set contains suitable columns, a histogram or boxplot is shown below._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    if numeric_field in df:
        sns.histplot(df[numeric_field], bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()
    else:
        print(f"No numeric field '{numeric_field}' found for visualization.")
    
    # Boxplot by group, if available
    if 'group_field' in locals() and group_field in df and numeric_field in df:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion

In this notebook, you've learned how to:

- Load a Croissant dataset with `mlcroissant`
- Enumerate record sets, fields, and columns, using their `@id` values
- Extract tabular data for analysis
- Perform basic exploratory data analysis (EDA)
- Visualize the distribution of numeric data

This template can be extended to perform deeper analyses, including feature engineering, advanced visualization, and modeling, according to your research needs.